# BitRoss 2 · 2/3 · Caption with a vision-language model

Captions textures with **Qwen3-VL-4B-Instruct** (FP8) through vLLM: a clean name, a one-line visual
caption and a 1-5 quality score. Shards are written to Drive as they finish, so you can stop and
resume any time (finished shards are skipped).

**Runtime:** L4 GPU, about 30 images/s. By default it labels all items plus blocks whose file names
are meaningless (≈350K images, ≈3 h); named blocks keep their filename + color captions. Use
`SUBSET = "all"` to caption everything (≈6 h), or `Qwen/Qwen3-VL-2B-Instruct` for ~2x speed.

In [ ]:
#@title Setup: Drive, code, Hugging Face login
#@markdown Put your Hugging Face token in Colab **Secrets** (key icon in the left bar) as `HF_TOKEN`
#@markdown and enable notebook access. The dataset is private, so the token is required.
REPO_URL = "https://github.com/OVAWARE/BitRoss"  #@param {type:"string"}
BRANCH = "v2"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/BitRoss2"  #@param {type:"string"}

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists("/content/BitRoss"):
    code_zip = f"{DRIVE_ROOT}/code.zip"
    if os.path.exists(code_zip):            # fallback: upload a zip of the repo to Drive
        subprocess.run(["unzip", "-q", code_zip, "-d", "/content/BitRoss"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/BitRoss"], check=True)
os.chdir("/content/BitRoss")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title Install vLLM in its own venv (keeps Colab's torch intact) and fetch the work dir
!pip -q install uv
!uv venv -q -p 3.12 /content/vllm-env
!uv pip install -q --python /content/vllm-env/bin/python vllm pandas pyarrow scipy pillow
!uv pip install -q --python /content/vllm-env/bin/python --no-deps -e /content/BitRoss
!mkdir -p /content/work && cp {DRIVE_ROOT}/work/pixels.npy {DRIVE_ROOT}/work/filtered.parquet {DRIVE_ROOT}/work/palette.json /content/work/
VPY = "/content/vllm-env/bin/python"

In [ ]:
#@title Quick check on 200 images before the long run
!{VPY} -m bitross.data.vlm_label --work /content/work --out /content/label_test --limit 500 --shard-size 500 --quantization fp8 2>&1 | grep -E "shard|example"
!head -n 15 /content/label_test/labels_0000.jsonl

In [ ]:
#@title Full run (resumable; re-run this cell after a disconnect)
MODEL = "Qwen/Qwen3-VL-4B-Instruct"  #@param ["Qwen/Qwen3-VL-4B-Instruct", "Qwen/Qwen3-VL-2B-Instruct", "Qwen/Qwen2.5-VL-3B-Instruct"]
SUBSET = "items_and_unnamed"  #@param ["items_and_unnamed", "all"]
!{VPY} -m bitross.data.vlm_label --work /content/work --out {DRIVE_ROOT}/labels --model {MODEL} --quantization fp8 \
    --batch 2048 --shard-size 10000 --subset {SUBSET} 2>&1 | grep -E "images,|^shard"

In [ ]:
#@title Build the training cache with VLM captions merged in
!python -m bitross.data.build cache --work /content/work --out /content/cache --vlm-labels "{DRIVE_ROOT}/labels/**/*.jsonl"
!mkdir -p {DRIVE_ROOT}/cache && cp -r /content/cache/* {DRIVE_ROOT}/cache/